# Import Mount Isa from OpenStreetMap

This demo keeps two local input options. The default uses OSMnx to pre-download and cache OSM responses in python/data/osm/osmnx_cache, then AequilibraE imports from that cache. After the first download, reuse or share that directory for a fast repeatable demo. The alternate option imports a prepared .osm.pbf file through pyrosm.

For the OSMnx cache option, no third-party extract service is used: OSMnx fetches the compact Mount Isa study area from OpenStreetMap services. The saved cache contains OSM HTTP response JSON, not a .osm.pbf file. Keep the cache directory and use the same study area, OSMnx version, and request settings when replaying it.

The live area is centered on the top-ranked Mount Isa town result with a 5 km radius, avoiding the much larger administrative boundaries returned by a full place-name search. OSM data is community maintained; retain attribution and review coverage before analysis.


In [ ]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import geopandas as gpd
import pandas as pd
from aequilibrae import Project

# A fresh folder lets the notebook be run more than once without replacing an existing project.
project_path = Path(gettempdir()) / f"aequilibrae_mount_isa_{uuid4().hex[:8]}"
print(f"AequilibraE project folder: {project_path}")


In [ ]:
import logging

from aequilibrae.utils import (
    basic_config,
    ensure_spatialite_binaries,
    set_known_spatialite_folder,
)

logger = logging.getLogger("aequilibrae")
basic_config(logging.DEBUG)

In [ ]:
data_dir = Path.cwd() / "data"
assert data_dir.is_dir()

## Download and crop the Queensland PBF

This downloads the current Geofabrik Queensland extract, then uses Pyrosm to write a Mount Isa PBF crop. The extent matches the coordinates in the osmium example below; Pyrosm expects `[west, south, east, north]`, so the two latitude values are ordered from south to north. The source and crop are kept in `python/data/osm` (or `data/osm` when the notebook is launched from `python`).


In [ ]:
from urllib.request import urlretrieve

from pyrosm import OSM

osm_data_dir = data_dir / "osm"

geofabrik_url = "https://download.geofabrik.de/australia-oceania/australia/queensland-latest.osm.pbf"
source_pbf_path = osm_data_dir / "queensland-latest.osm.pbf"
pbf_path = osm_data_dir / "mount-isa.osm.pbf"

bbox = [139.45510, -20.79793, 139.57898, -20.60647]  # west, south, east, north
if not pbf_path.is_file():
    print(f"Downloading {geofabrik_url}")
    urlretrieve(geofabrik_url, source_pbf_path)
    print(f"Cropping Queensland PBF to {bbox}")
    OSM(source_pbf_path, bounding_box=bbox).to_pbf(
        output_path=str(pbf_path), repack=True
    )

print(f"Mount Isa PBF: {pbf_path}")

## Import the cached data into AequilibraE
  With osmnx_cache, the AequilibraE importer repeats the same polygon query and OSMnx serves it from osmnx_cache. Once the cache has been prepared, this can run without reaching Nominatim or Overpass. With pbf, the importer reads the local file through pyrosm.


In [ ]:
project = Project.create(project_path)

In [ ]:
project.tables.importer.osm(
    pbf_path=pbf_path,
    modes=("car", "transit", "bicycle", "walk"),
    simplify=True,
)

## Inspect the imported network

The project keeps its network in the AequilibraE project database. Inspect the imported link and node tables and check the modes represented in the network.

In [ ]:
links = project.tables.links.data
nodes = project.tables.nodes.data
print(f"Links: {len(links):,}\nNodes: {len(nodes):,}")
links.head()

In [ ]:
zones_file = data_dir / "Mount Isa SA1.parquet"
employment_file = data_dir / "Mount Isa SA1 employment.csv"

In [ ]:
zones = gpd.read_parquet(zones_file)[["SA1_CODE_2021", "geometry"]].explode()
zones["SA1_CODE_2021"] = zones["SA1_CODE_2021"].astype(int)
zones = zones.rename(columns={"SA1_CODE_2021": "sa1_code_2021"}).set_index(
    "sa1_code_2021"
)
print(f"Zones: {len(zones):,}")
zones.head()

In [ ]:
employment = (
    pd.read_csv(employment_file, skiprows=9)  # Skip ABS header
    .tail(-1)  # Drop the first row which is blank for the two row header
    .head(-5)  # Drop the last 5 rows which contain data source info
    .loc[:, :"Total"]  # Drop columns after "Total", just whitespace
    .rename(
        columns={"1-digit level INDP Industry of Employment": "SA1_CODE_2021"}
    )  # Pandas reads this column name incorrectly due to the multi-row header
    .astype(int)
)

column_names = {
    column: column.lower().replace(" ", "_").replace(",", "")
    for column in employment.columns
} | {"SA1_CODE_2021": "sa1_code_2021"}

employment = employment.rename(columns=column_names).set_index("sa1_code_2021")

print(f"Employment: {len(employment):,}")
employment.head()

In [ ]:
zones = zones.join(employment, how="left")

Our dataframe has more columns than our database! Before we store our new zoning system we should create the columns we'd like to keep. This can be done through plan SQLite queries, or through the field editor.

In [ ]:
existing_columns = set(project.tables.zones.columns)
columns = set(zones.columns)

missing_columns = columns - existing_columns
print("Missing columns:", missing_columns)

In [ ]:
import sqlite3

try:
    with project.db_connection as conn:
        conn.execute('ALTER TABLE zones ADD COLUMN "SA1_CODE_2021" INTEGER')
except sqlite3.OperationalError as e:
    print("Error adding column:", e)

In [ ]:
fields = project.tables.zones.fields
existing_fields = fields.all_fields()
print("Existing zone fields:", existing_fields)

for proper_name, column in column_names.items():
    if column not in existing_fields:
        fields.add(column, description=f"ABS: {proper_name}", data_type="integer")
        logger.info("Added field: %s", column)

fields.save()

We'll table the zones dataframe that we've constructed and insert it into the database. Because we haven't supplied or set the ``zone_ids`` column ourselves, AequilibraE will infer that from the insertion order. If you need control over the IDs, say for instance, to match some existing IDs, then you should set that manually.

In [ ]:
zones["zone_id"] = project.tables.zones.insert_from(zones.reset_index())

Let's now add centroids to each of our zones, and connect them to the ``b`` mode  links with centroid connectors.

In [ ]:
project.tables.zones.add_centroids()
project.tables.zones.connect_mode("b")

## Simple example

In [ ]:
from aequilibrae.route_choice import RouteChoice

Here we'll build the road network from the project for a subset of modes. We're only interested in the ``b`` mode here.

In [ ]:
network = project.road.build_network(modes=["b"])

We'll then build the graph ourselves to provide to ``RouteChoice``. We'll use the ``b`` mode and the ``distance`` field as the cost function.

In [ ]:
graph = network.build_graph(
    mode="b",
    cost="distance",
    compact=True,
    remove_dead_ends=True,
    block_centroid_flows=False,
)

For this small example we'll examine the route choice sets of the following OD pair.

In [ ]:
od_pair = (14, 37)

We can then construct the ``RouteChoice`` object and set our parameters. Here we're using link penalisation with a maximum of 10 routs.

In [ ]:
choice = RouteChoice(graph)
choice.set_choice_set_generation(
    algorithm="link-penalisation",
    max_routes=10,
    max_depth=12,
    disutility_cutoff_coefficient=1.05,
    disutility_cutoff_constant=0,
)

Because we're not providing a demand matrix here we'll call ``prepare`` with the set of OD pairs to compute paths for. If we'd like the path sized logit and probablity figures for the resulting routes we can set ``perform_assignment=True``, if we'd just like the routes themselves that can be set to ``False``.  

In [ ]:
choice.prepare([od_pair])
choice.execute(perform_assignment=True)
choice.get_results()

## Exploring the route choice sets

In this advanced example we'll use the lonboard visulaisation library to create an interactive demo of the resulting routes.

### Build the bicycle routing graph

`BicycleRouteChoice` loads the bicycle network from the project, including its centroids and turn restrictions. It builds a compact graph using link distance in metres as the cost.

See [route_choice_workshop.py](route_choice_workshop.py) for the AequilibraE code. You can inspect the two result tables with `routing.compare((14, 37))` using just the routing module.

In [ ]:
from route_choice_workshop import BicycleRouteChoice

routing = BicycleRouteChoice(project)

Choose an origin and destination and click **Run comparison**. Both tabs use the same trip. BFSLE searches by removing links from routes it has found. LP searches again after increasing the cost of used links. Both use the same distance costs, route cap and 5% cutoff for inclusion. Each method applies the search depth limit differently.

- **BFSLE vs LP** shows the two choice sets with linked map views. Use **Compare link use** to highlight links shared by both methods and links found by just one, counting included routes in either direction.
- **Close a link** lets you click a bicycle road or enter its link ID. Click **Close link** to regenerate the choice set with that road closed in both directions. The closed link is red. **Reopen link** restores the original results.

Use **Route** to inspect an alternative, including excluded routes. The shortest route sits on top. Green marks the origin and purple marks the destination. Use **Fit zones** to see the whole zoning system. Hover inside a zone to see its centroid ID and SA1 code.

The maps and controls are in [route_choice_visualisation.py](route_choice_visualisation.py).

In [ ]:
from route_choice_visualisation import route_choice_workshop

# Refresh the map data after adding centroid connectors.
links = project.tables.links.data
nodes = project.tables.nodes.data

route_choice_workshop(routing, links, nodes, zones)